# Transfer learning

This is the transfer learning notebook, used for the second test of the project.

The first half of the code is near identical to the baseline version

In [ ]:
import os
#import cv2
import glob
import keras
import random
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from keras._tf_keras.keras.models import load_model
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from keras._tf_keras.keras.callbacks import ModelCheckpoint, EarlyStopping
from keras._tf_keras.keras.layers import BatchNormalization,MaxPooling2D,Flatten
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, precision_score, recall_score
from keras._tf_keras.keras.layers import Conv2D,Dense,Dropout,GlobalAveragePooling2D
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import logging
# Set TensorFlow log level to only display errors
tf.get_logger().setLevel(logging.ERROR)

In [ ]:
path="flowers"
categories=os.listdir(path)
print(categories)

In [ ]:
def create_image_dataframe(dataset_dir):
    """
    Creates a DataFrame with image paths and labels from a dataset directory.
    """
    data = []
    
    image_extensions = ['*.png', '*.jpg', '*.jpeg',]
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(dataset_dir, '**', ext), recursive=True))
    
    for path in image_paths:
        label = os.path.basename(os.path.dirname(path))
        data.append((label, path))
    
    df = pd.DataFrame(data, columns=['label', 'path'])
    return df

In [ ]:
df=create_image_dataframe(path)
df.head()

In [ ]:
num_images=9
# Randomly sample num_images from the DataFrame
sampled_df = df.sample(n=num_images).reset_index(drop=True)

plt.figure(figsize=(7, 5))

for i in range(num_images):
    plt.subplot(3, 3, i + 1)

    # Read the image
    img_path = sampled_df.iloc[i]['path']
    img = plt.imread(img_path)

    plt.imshow(img)
    plt.title(sampled_df.iloc[i]['label'])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
counts=df['label'].value_counts()
counts

In [ ]:
unique_labels=['iris', 'dandelion', 'tulip', 'magnolia', 'coreopsis', 'sunflower', 
 'california_poppy', 'black_eyed_susan', 'rose', 'water_lily', 'common_daisy', 
 'calendula', 'daffodil', 'carnation', 'bellflower', 'astilbe']
plt.bar(unique_labels, counts, width=0.8)  

plt.xlabel('The type of sentiment')
plt.ylabel('Number of the Samples')
plt.title("The Number of Samples for each sentiment")
plt.xticks(rotation=45, ha='right') 
plt.tight_layout() 
plt.show()

In [ ]:
#uncomment the batch that you want to use
#FIRST BATCH
selected_labels=['common_daisy', 'bellflower', 'rose', 'sunflower', 'astilbe', 'black_eyed_susan', 'dandelion', 'carnation', 'water_lily', 'magnolia']
rare_samples=['iris', 'tulip', 'coreopsis', 'california_poppy', 'calendula', 'daffodil']

#SECOND BATCH
#selected_labels=['iris', 'tulip', 'coreopsis', 'california_poppy', 'calendula', 'daffodil', 'common_daisy', 'bellflower', 'rose', 'sunflower']
#rare_samples=['astilbe', 'black_eyed_susan', 'dandelion', 'carnation', 'water_lily', 'magnolia']

#THIRD BATCH
#selected_labels=['astilbe', 'black_eyed_susan', 'dandelion', 'daffodil', 'water_lily', 'magnolia', 'iris', 'tulip', 'california_poppy', 'calendula']
#rare_samples=['common_daisy', 'bellflower', 'rose', 'sunflower', 'coreopsis', 'carnation']

#FOURTH BATCH
#selected_labels=['astilbe', 'water_lily', 'iris', 'calendula', 'common_daisy', 'rose', 'carnation', 'dandelion', 'california_poppy', 'coreopsis']
#rare_samples=['black_eyed_susan', 'magnolia', 'tulip', 'daffodil', 'bellflower', 'sunflower']

#FIFTH BATCH
#selected_labels=['black_eyed_susan', 'bellflower', 'calendula', 'coreopsis', 'daffodil', 'iris', 'magnolia', 'rose', 'tulip', 'water_lily']
#rare_samples=['astilbe', 'california_poppy', 'carnation', 'common_daisy', 'dandelion','sunflower']

import pandas as pd

def select_and_sample_classes(df, selected_labels, rare_samples, num_samples=700, rare_num=20):

    # Filter the DataFrame for selected labels
    filtered_df = df[df['label'].isin(selected_labels)]
    
    sampled_df = pd.DataFrame()
    for label in selected_labels:
        label_df = filtered_df[filtered_df['label'] == label].sample(n=num_samples, replace=True)
        sampled_df = pd.concat([sampled_df, label_df], ignore_index=True)
    
    rare_df = df[df['label'].isin(rare_samples)]
    for label in rare_samples:
        label_df = rare_df[rare_df['label'] == label].sample(n=rare_num, replace=True)
        sampled_df = pd.concat([sampled_df, label_df], ignore_index=True)

    return sampled_df

def sample_common_classes(df, selected_labels, num_samples=700):

    filtered_df = df[df['label'].isin(selected_labels)]
    
    sampled_df = pd.DataFrame()
    for label in selected_labels:
        label_df = filtered_df[filtered_df['label'] == label].sample(n=num_samples, replace=True)
        sampled_df = pd.concat([sampled_df, label_df], ignore_index=True)

    return sampled_df

    

def sample_rare_classes(df, rare_samples, num_samples=700):

    sampled_df = pd.DataFrame()
    
    rare_df = df[df['label'].isin(rare_samples)]
    for label in rare_samples:
        label_df = rare_df[rare_df['label'] == label].sample(n=num_samples, replace=True)
        sampled_df = pd.concat([sampled_df, label_df], ignore_index=True)

    return sampled_df


#sampled_df = select_and_sample_classes(df, selected_labels, rare_samples, num_samples=700, rare_num=30)
#sampled_df['label'].value_counts()

base_df = sample_common_classes(df, selected_labels, num_samples=700)

common_df = sample_common_classes(df, selected_labels, num_samples=700) 
rare_df = sample_rare_classes(df, rare_samples, num_samples=700) 

common_df['label'].value_counts()
rare_df['label'].value_counts()

At this point, The baseline train test split is done, but the rare classes are ignored until after the pre-transfer training is complete

In [ ]:
#Base ver. split base_df into 70% train, 15% test, 15% valid
base_train, base_temp = train_test_split(base_df, test_size=0.3, random_state=42)
base_test, base_valid = train_test_split(base_temp, test_size=0.5, random_state=42)

print("Training set shapes:", base_train.shape)
print("Testing set shapes:", base_test.shape)
print("Validation set shapes:", base_valid.shape)

In [ ]:
train_datagen = ImageDataGenerator(    
    rescale=1.0/255,        
    shear_range=0.2,
    rotation_range=.1,
    zoom_range=0.2,       
    horizontal_flip=True ,
)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Training data generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=base_train,
    x_col='path', 
    y_col='label',   
    target_size=(224, 224), 
    batch_size=32,
    class_mode='categorical', 
    color_mode='rgb',
    shuffle=True
)

# Validation data generator
valid_generator = test_datagen.flow_from_dataframe(
    dataframe=base_valid,
    x_col='path', 
    y_col='label',
    target_size=(224, 224), 
    batch_size=32,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=False  
)

# Testing data generator
test_generator = test_datagen.flow_from_dataframe(
    dataframe=base_test,
    x_col='path', 
    y_col='label',  
    target_size=(224, 224),  
    color_mode='rgb',
    batch_size=64,  
    class_mode='categorical',
    shuffle=False  
)

In [ ]:
classes=list(train_generator.class_indices)
classes

In [ ]:
batch_size = 9
# Generate a batch of images and labels
images, labels = next(train_generator)
# Plot the images with their labels
plt.figure(figsize=(10, 10))
for i in range(min(len(images), 9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    label = np.argmax(labels[i])
    plt.title(f"Density: {classes[label]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model=keras.models.Sequential()

model.add(Conv2D(64,(3,3),input_shape=(224,224,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.1))


model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.2))

model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(10,activation='softmax')) #MAYBE 10?
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
weight_path = "cnn_weights.best.weights.h5"

checkpoint = ModelCheckpoint(weight_path, monitor='val_accuracy', verbose=1, 
                             save_best_only=True, mode='max', save_weights_only=True)

early = EarlyStopping(monitor="val_accuracy", 
                      mode="max", 
                      patience=10)

callbacks_list = [checkpoint, early]

In [ ]:
epoch_num=40
history=model.fit(train_generator,epochs=epoch_num,validation_data=(valid_generator),callbacks=callbacks_list)

In [ ]:
# load the best weights
model.load_weights(weight_path)

In [ ]:
# Evaluate the model on the test data
evaluation_result = model.evaluate(valid_generator)

# The result
print("Test Loss:", evaluation_result[0])
print("Test Accuracy:", evaluation_result[1])

In [ ]:
#  predictions on test data
y_pred = model.predict(valid_generator)

In [ ]:
y_pred_classes = np.argmax(y_pred, axis=1)
#  y_test is one-hot encoded
y_test_classes = valid_generator.classes



confusion_Matrix = confusion_matrix(y_test_classes, y_pred_classes)

# Plotting the confusion matrix as a heatmap
sns.heatmap(confusion_Matrix, annot=True, cmap='Purples', fmt='g')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# Plot the training and validation accuracy
ax1.plot(history.history['accuracy'])
ax1.plot(history.history['val_accuracy'])
ax1.set_title('Model accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend(['Train', 'Validation'], loc='upper left')

# Plot the training and validation loss
ax2.plot(history.history['loss'])
ax2.plot(history.history['val_loss'])
ax2.set_title('Model loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend(['Train', 'Validation'], loc='upper left')

# Display the plots
plt.show()

In [ ]:
print(classification_report(y_test_classes, y_pred_classes))

In [ ]:
x_test,y_test=next(test_generator)
# make a figure that images will be shown in
fig, axs = plt.subplots(3, 3, figsize=(7, 7))
axs = axs.flatten()

for i in range(9):
    #predict image
    predicted = model.predict(np.array([x_test[i]]))
    # take larger value which is predicted class
    predictedClass = np.argmax(predicted)
    actual = np.argmax(y_test[i])
    # plot actual class and predicted with the image
    axs[i].imshow(x_test[i], cmap='gray')
    axs[i].set_title(f'Predicted: {classes[predictedClass]} \nActual: {classes[actual]}')
    axs[i].axis('off')

plt.tight_layout()
plt.show()

This is where the baseline version would end. A copy is made of the baseline model so far

In [ ]:
# SAVE CURRENT MODEL SO IT DOESNT GET DELETED
Temp_model = model

Temp_model.load_weights(weight_path)
Temp_model.summary()

At this point, the top layer of the CNN is removed and replaced with one used to classify both common and rare plants. Some number of layers are frozen, change the number in the for loop definition to change the number of frozen layers.

In [ ]:
Temp_model.pop()
for layer in Temp_model.layers[:128]:
    layer.trainable = False
Temp_model.add(Dense(16, activation='softmax'))

Temp_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
Temp_model.summary()

In [ ]:
rare_df = sample_rare_classes(df, rare_samples, num_samples=180)
rare_valid, rare_temp = train_test_split(rare_df, test_size=1/6, random_state=42)
rare_train, rare_test = train_test_split(rare_temp, test_size=0.5, random_state=42)

train_df = pd.concat([base_train, rare_train], ignore_index=True)
test_df = pd.concat([base_test, rare_test], ignore_index=True)
valid_df = pd.concat([base_valid, rare_valid], ignore_index=True)


# NEW DATA GENERATORS
# Training data generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path', 
    y_col='label',   
    target_size=(224, 224), 
    batch_size=32,
    class_mode='categorical', 
    color_mode='rgb',
    shuffle=True
)

# Validation data generator
valid_generator = test_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col='path', 
    y_col='label',
    target_size=(224, 224), 
    batch_size=32,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=False  
)

# Testing data generator
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path', 
    y_col='label',  
    target_size=(224, 224),  
    color_mode='rgb',
    batch_size=64,  
    class_mode='categorical',
    shuffle=False  
)

In [ ]:
weight_path = "cnn_weights_2.best.weights.h5"

checkpoint = ModelCheckpoint(weight_path, monitor='val_accuracy', verbose=1, 
                             save_best_only=True, mode='max', save_weights_only=True)

early = EarlyStopping(monitor="val_accuracy", 
                      mode="max", 
                      patience=10)

callbacks_list = [checkpoint, early]

In [ ]:
epoch_num=40
history=Temp_model.fit(train_generator,epochs=epoch_num,validation_data=(valid_generator),callbacks=callbacks_list)

In [ ]:
# load the best weights
Temp_model.load_weights(weight_path)

In [ ]:
# Evaluate the model on the test data
evaluation_result = Temp_model.evaluate(valid_generator)

# The result
print("Test Loss:", evaluation_result[0])
print("Test Accuracy:", evaluation_result[1])

In [ ]:
#  predictions on test data
y_pred = Temp_model.predict(valid_generator)

In [ ]:
y_pred_classes = np.argmax(y_pred, axis=1)
#  y_test is one-hot encoded
y_test_classes = valid_generator.classes
new_classes = list(train_generator.class_indices)
print(new_classes)



confusion_Matrix = confusion_matrix(y_test_classes, y_pred_classes)

# Plotting the confusion matrix as a heatmap
sns.heatmap(confusion_Matrix, annot=True, cmap='Purples', fmt='g')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.show()
#BF BAS DAF MAG SF TULIP

#BF PREDICTED AS IRIS (should put these together)
#BAS PREDICTED AS CORE (VERY OFTEN, PUT THESE TOGETHER)
#DAF PREDICTED AS CORE/CALE
#MAG PREDICTED AS WL (kinda often)
#SF PREDUCTED AS CORE
#T PREDICTED AS ROSE

#BAS BF CALE CORE DAF IRIS MAG ROSE T WL

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
# Plot the training and validation accuracy
ax1.plot(history.history['accuracy'])
ax1.plot(history.history['val_accuracy'])
ax1.set_ylim(ymin=0)
ax1.set_title('Model accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend(['Train', 'Validation'], loc='upper left')

# Plot the training and validation loss
ax2.plot(history.history['loss'])
ax2.plot(history.history['val_loss'])
ax2.set_title('Model loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend(['Train', 'Validation'], loc='upper left')

# Display the plots
plt.show()